## Environment Setup
Do not modify the following two cells. __adh_init() details here: [README-adh_init](https://github.com/cfacorp/aao-dbx-helpers/blob/master/README-adh_init.md).

In [0]:
import os
if os.path.isfile("shared/frozen_requirements.txt"):
    %pip install -r shared/frozen_requirements.txt # NOQA: E999
    dbutils.library.restartPython()

In [0]:
import os
from importlib.machinery import SourceFileLoader

__aadhf = SourceFileLoader(
    "aao_dbx_helpers_finder",
    "/Workspace/Repos/global/aao-dbx-helpers/.boot/finder.py",
).load_module()
__aadhf.find_aao_dbx_helpers()

from aao_dbx_helpers import adh_init as __adh_init

__adh_init()


## Task Code

In [0]:
%sh
pip install geopy pygeohash

In [0]:
import geopy.distance
import pygeohash as pgh
import pandas as pd
from datetime import datetime
from collections import defaultdict
import boto3
from redshift import load_from_redshift


# read store characteristics	
query = """
SELECT
	location_number AS location_num,
	opened_dt AS open_date,
	restaurant_type AS location_type_code,
	latitude,
	longitude,
    DATE_PART_YEAR(open_date) AS year
FROM conf.location_attributes 
WHERE 1=1
    AND status IN ('open', 'remodel')
    AND year >= 2017
"""
location_attributes = load_from_redshift(query)
loc = location_attributes.toPandas()

In [0]:
display(loc)

In [0]:
newStoreImpact_ = defaultdict(dict)
precision = 2

max_distance = 150
for row in loc.values:
    l, o, t, lat, lon, y = row
    geohash1 = pgh.encode(latitude=lat, longitude=lon, precision=precision)
    if t in ['Stand Alone']:
        for row_ in loc.values:
            l_, o_, t_, lat_, lon_, y_ = row_
            geohash2 = pgh.encode(latitude=lat_, longitude=lon_, precision=precision)
            if l != l_ and o_ < o and t_ in ['Stand Alone'] and geohash1 == geohash2:
                try:
                    distance = geopy.distance.distance((lat, lon), (lat_, lon_)).km
                    #print(l_, l, lat, lon, lat_, lon_, distance)
                    if distance < max_distance:
                        #print('added')
                        newStoreImpact_[l_][l] = {"distance": distance, "date": o, "loc_type": t}
                except:
                    continue
					
# import json
# with open("opendate_distance_history_test.json", "w") as f:
#     json.dump(newStoreImpact_, f)